In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

In [3]:
#sc.stop() # to ensure that this is the context being executed
credentials_location = '/Users/mafalda.rodrigues/.gc/keys/my-creds.json'


conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('test') \
    .set("spark.jars", "./lib/gcs-connector-hadoop3-2.2.5.jar") \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)

In [4]:
sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl",  "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

25/04/18 17:03:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/18 17:03:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [11]:
df = spark.read.csv('gs://public-life-data-seattle-bucket/*.csv', header=True, inferSchema=True)


In [12]:
df.count()

17281

In [ ]:
# Escrevendo no BigQuery na tabela public_life_data_seattle
df.write \
    .format("bigquery") \
    .option("table", "project-zoomcamp-457121.public_life_data_seattle_dataset.public_life_data_seattle") \
    .option("temporaryGcsBucket", "public-life-data-seattle-bucket") \  # Seu bucket existente
    .mode("overwrite") \
    .save()